# Incremental Data Pipeline: 
FMCG Staging & Fact Ingestion (Medallion Architecture)This notebook implements an automated Incremental Ingestion & Staging Pipeline following the Medallion Architecture (Bronze Staging $\rightarrow$ Silver Staging $\rightarrow$ Gold Fact) using PySpark and Delta Lake on Databricks. It includes data cleansing, quality checks, automated staging cleanup (TRUNCATE), and additive aggregation merges.

In [0]:
# Import PySpark window functions, standard functions, and Delta table modules
from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "../Utitlites"

In [0]:
# Create notebook widgets
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

# Extract parameters correctly
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(f"Target Catalog: {catalog}")
print(f"Data Source: {data_source}")

Target Catalog: fmcg
Data Source: orders


## Import Data From Landing File

In [0]:
# Read raw CSV file and attach metadata attributes
staging_bronze = (spark.read.format('csv')
        .option('header', True)
        .option('inferSchema', True)
        .load(f'{incremental_load_order_path}/*.csv')
        .withColumn('read_timestamp', F.current_timestamp())
        .select('*', '_metadata.file_name', '_metadata.file_size')
)

# Display Schema
staging_bronze.printSchema()

# Dispay Sample
staging_bronze.show(10)

root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)

+-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|     order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FDEC818102602|Tuesday, December...|     789102|  25891502|    196.0|2026-10-08 11:04:...|orders_2025_12_16...|    23060|
|FDEC818102602|Tuesday, December...|     789102|  25891503|     NULL|2026-10-08 11:04:...|orders_2025_12_16...|    23060|
|FDEC818102602|Tuesday, December.

## Upload Data For Staging Bronze

In [0]:
table_name = f"{catalog}.bronze.sb_fact_staging_{data_source}"

staging_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", True) \
    .saveAsTable(table_name)

# Show Table
staging_bronze = spark.sql(f"SELECT * FROM {table_name}")
display(spark.sql(f"SELECT * FROM {table_name} LIMIT 10"))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FDEC83622503,"Monday, December 01, 2025",789622,25891302,39.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062
FDEC83622503,"Monday, December 01, 2025",789622,25891301,26.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062
FDEC83622503,"Monday, December 01, 2025",789622,25891503,205.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062
FDEC83622503,"Monday, December 01, 2025",789622,25891501,184.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062
FDEC83622503,"Monday, December 01, 2025",789622,25891101,327.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062
FDEC83622503,"Monday, December 01, 2025",INVALID,25891401,480.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062
FDEC83622503,"Monday, December 01, 2025",INVALID,25891401,480.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062
FDEC83622503,"Monday, December 01, 2025",789622,25891301,26.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062
FDEC83622503,01-12-2025,789622,25891503,205.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062
FDEC82301603,"Monday, December 01, 2025",789301,25891302,54.0,2026-10-08T11:04:10.815Z,orders_2025_12_01.csv,21062


## Cleaning Stage

In [0]:
# Read from Bronze Staging
df_clean = spark.read.table(table_name)

# Filter null order IDs
df_clean = df_clean.filter(F.col("order_id").isNotNull())

# Clean Entity IDs (product_id & customer_id)
entity_cols = ['product_id', 'customer_id']
for col_name in entity_cols:
    df_clean = df_clean.withColumn(
        col_name,
        F.when(F.col(col_name).rlike('^[0-9]+$'), F.col(col_name)).otherwise('999999')
    )

# Enable legacy time parser for flexibility
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# Normalize Date Formats
df_clean = df_clean.withColumn("order_placement_date", F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", ""))

months_map = {
    "January": "1", "February": "2", "March": "3", "April": "4",
    "May": "5", "June": "6", "July": "7", "August": "8",
    "September": "9", "October": "10", "November": "11", "December": "12"
}

for month_name, month_num in months_map.items():
    df_clean = df_clean.withColumn("order_placement_date", F.regexp_replace(F.col("order_placement_date"), month_name, month_num))

df_clean = df_clean.withColumn("order_placement_date", F.regexp_replace(F.col("order_placement_date"), r"\s*,\s*", "/")) \
                   .withColumn("order_placement_date", F.regexp_replace(F.col("order_placement_date"), r"\s+", "/"))

df_clean = df_clean.withColumn(
    "order_placement_date",
    F.coalesce(
        F.expr("try_to_date(order_placement_date, 'M/d/yyyy')"),
        F.expr("try_to_date(order_placement_date, 'd/M/yyyy')"),
        F.expr("try_to_date(order_placement_date, 'yyyy/M/d')"),
        F.expr("try_to_date(order_placement_date, 'd-M-yyyy')"),
        F.expr("try_to_date(order_placement_date, 'yyyy-M-d')")
    )
)

# Filter null quantities and remove duplicates
df_clean = df_clean.filter(F.col('order_qty').isNotNull()).dropDuplicates()

# Join with Product Dimension table to retrieve product_code
df_products = spark.sql(f'SELECT product_id, product_code FROM {catalog}.gold.sb_dim_products')
df_clean = df_clean.join(df_products, on='product_id', how='inner')

df_clean.show(10)

+----------+------------+--------------------+-----------+---------+--------------------+--------------------+---------+--------------------+
|product_id|    order_id|order_placement_date|customer_id|order_qty|      read_timestamp|           file_name|file_size|        product_code|
+----------+------------+--------------------+-----------+---------+--------------------+--------------------+---------+--------------------+
|  25891203|FDEC83621601|          2025-01-12|     789621|    416.0|2026-10-08 11:04:...|orders_2025_12_01...|    21062|91a67435e55806eae...|
|  25891203|FDEC82401602|          2025-12-01|     789401|    293.0|2026-10-08 11:04:...|orders_2025_12_01...|    21062|91a67435e55806eae...|
|  25891202|FDEC82220202|          2025-12-01|     999999|    364.0|2026-10-08 11:04:...|orders_2025_12_01...|    21062|96b71be076213a8ec...|
|  25891302|FDEC83320403|          2025-12-01|     789320|     83.0|2026-10-08 11:04:...|orders_2025_12_01...|    21062|d9ebd1ca64d23951a...|
|  258

### Check Data Quality From Cleaning Stage 

In [0]:
# ==========================================
# Data Quality Check (Print Only)
# ==========================================

# 1. حساب إحصائيات الجودة
total_records = spark.read.table(table_name).count()
valid_records = df_clean.count()
rejected_records = total_records - valid_records
success_rate = round((valid_records / total_records * 100), 2) if total_records > 0 else 0.0

# 2. عرض تقرير الجودة
print("================ DATA QUALITY REPORT ================")
print(f" Data Source: {data_source}")
print(f" Total Input Records (Bronze): {total_records}")
print(f" Valid Records (Cleaned):     {valid_records}")
print(f" Rejected Records:            {rejected_records}")
print(f" Quality Success Rate:        {success_rate}%")
print("=====================================================")

================ DATA QUALITY REPORT ================
 Data Source: orders
 Total Input Records (Bronze): 9967
 Valid Records (Cleaned):     7878
 Rejected Records:            2089
 Quality Success Rate:        79.04%


## Silver & Staging Silver Processig
Appends or merges cleaned batch records into the Silver staging table and reports row counts before and after insertion.

In [0]:
# Target Silver Staging Table Name
stag_silver_name = f'{catalog}.silver.sb_fact_staging_{data_source}'

# Perform Append (Initial Load) or Merge (Incremental)
if not spark.catalog.tableExists(stag_silver_name):
    df_clean.write \
        .format('delta') \
        .mode('append') \
        .option('delta.enableChangeDataFeed', 'true') \
        .saveAsTable(stag_silver_name)
    count_before = 0
    count_after = spark.sql(f"SELECT COUNT(*) FROM {stag_silver_name}").collect()[0][0]
else:
    count_before = spark.sql(f"SELECT COUNT(*) FROM {stag_silver_name}").collect()[0][0]
    
    stag_silver_target = DeltaTable.forName(spark, stag_silver_name)
    stag_silver_target.alias('target').merge(
        source=df_clean.alias('source'),
        condition='''
            target.order_placement_date = source.order_placement_date 
            AND target.order_id = source.order_id 
            AND target.product_code = source.product_code 
            AND target.customer_id = source.customer_id
        '''
    ) \
    .whenNotMatchedInsertAll() \
    .execute()
    
    count_after = spark.sql(f"SELECT COUNT(*) FROM {stag_silver_name}").collect()[0][0]

# Clear Staging Bronze table after successful Silver insert
spark.sql(f'TRUNCATE TABLE {table_name}')

print(f"[1] Staging Bronze ({table_name}) cleared successfully! \n")
print(f"--- Rows before merge to Silver: {count_before} ---")
print(f"--- Rows after merge to Silver: {count_after} (Added: {count_after - count_before}) ---")

[1] Staging Bronze (fmcg.bronze.sb_fact_staging_orders) cleared successfully! 

--- Rows before merge to Silver: 7878 ---
--- Rows after merge to Silver: 7878 (Added: 0) ---


## Gold Processing

## [1] Processing For sb_fact_orders on Gold

In [0]:
# Target Gold Fact Table Name
sb_fact_orders_gold_name = f'{catalog}.gold.sb_fact_{data_source}'

# 1. Prepare Schema & Align Attribute Names
df_fact_orders_silver = (
    df_clean.select([
        "order_id", 
        "order_placement_date", 
        "customer_id", 
        "product_code", 
        "order_qty"
    ])
    .withColumnRenamed('customer_id', 'customer_code')
    .withColumnRenamed('order_placement_date', 'date')
    .withColumnRenamed('order_qty','sold_quantity')
)

# 2. Append or Additive MERGE Into Gold Table
if not spark.catalog.tableExists(sb_fact_orders_gold_name):
    df_fact_orders_silver.write \
        .format('delta') \
        .mode('append') \
        .saveAsTable(sb_fact_orders_gold_name)
else:
    gold_sb_target = DeltaTable.forName(spark, sb_fact_orders_gold_name)
    
    gold_sb_target.alias('target').merge(
        source=df_fact_orders_silver.alias('source'),
        condition='''
            target.order_id = source.order_id
            AND target.product_code = source.product_code
            AND target.customer_code = source.customer_code
            AND target.date = source.date
        '''
    ) \
    .whenMatchedUpdate(set={
        'sold_quantity': 'target.sold_quantity + source.sold_quantity'
    }) \
    .whenNotMatchedInsertAll() \
    .execute()

print("Pipeline process completed successfully!")

Pipeline process completed successfully!


### Upsert for Parent Gold Table

In [0]:
# Prepare Monthly Sales Aggregations
df_monthly_agg = (
    df_clean
    .withColumn('date', F.trunc('order_placement_date', 'MM')) 
    .groupBy(['date', 'customer_id', 'product_code'])
    .agg(F.sum('order_qty').alias('sold_quantity'))
    .withColumnRenamed('customer_id', 'customer_code')
)

parent_fact_table = f'{catalog}.gold.fact_{data_source}'

# Save or Merge into Gold Parent Fact Table
if not spark.catalog.tableExists(parent_fact_table):
    df_monthly_agg.write.format('delta').mode('append').saveAsTable(parent_fact_table)
else:
    gold_parent_target = DeltaTable.forName(spark, parent_fact_table)
    gold_parent_target.alias('target').merge(
        source=df_monthly_agg.alias('source'),
        condition='''
            target.product_code = source.product_code
            AND target.customer_code = source.customer_code
            AND target.date = source.date
        '''
    ) \
    .whenMatchedUpdate(set={'sold_quantity': 'target.sold_quantity + source.sold_quantity'}) \
    .whenNotMatchedInsertAll() \
    .execute()

print("Pipeline process completed successfully!")

Pipeline process completed successfully!


In [0]:
# ==============================================================================
# GOLD LAYER DATA QUALITY AUDIT (FULL COMPOSITE KEYS CHECK)
# ==============================================================================
from pyspark.sql import functions as F

print("=" * 75)
print(" STARTING FULL COMPOSITE KEY QUALITY AUDIT FOR GOLD LAYER TABLES")
print("=" * 75)

# 1. Read Gold tables directly from Catalog
# ------------------------------------------------------------------------------
gold_sb_fact = spark.read.table(f'{catalog}.gold.sb_fact_{data_source}')
gold_parent_fact = spark.read.table(f'{catalog}.gold.fact_{data_source}')

# 2. AUDIT PART A: Detailed Gold Fact Table (sb_fact_orders)
# Full Composite Key: order_id + product_code + customer_code + date
# ------------------------------------------------------------------------------
sb_full_composite_keys = ["order_id", "product_code", "customer_code", "date"]
sb_total = gold_sb_fact.count()

# Check for Null values in any component of the composite key
sb_nulls = gold_sb_fact.filter(
    F.col("order_id").isNull() | 
    F.col("product_code").isNull() | 
    F.col("customer_code").isNull() | 
    F.col("date").isNull()
).count()

# Check for duplicate records across the full composite key
sb_duplicates_df = (
    gold_sb_fact
    .groupBy(sb_full_composite_keys)
    .count()
    .filter(F.col("count") > 1)
)
sb_duplicates = sb_duplicates_df.count()

print(f"\n[AUDIT 1] Detailed Gold Fact Table (gold.sb_fact_{data_source}):")
print(f"  |- Total Gold Records: {sb_total}")
print(f"  |- Nulls in Composite Key: {sb_nulls}")
print(f"  |- Full Composite Key Duplicates ({' + '.join(sb_full_composite_keys)}): {sb_duplicates}")

if sb_duplicates > 0:
    print("  WARNING: True Duplicates detected in Detailed Gold Table!")
    sb_duplicates_df.orderBy(F.col("count").desc()).show(5)
else:
    print("  PASSED: No true duplicates found across full composite transaction keys.")

# 3. AUDIT PART B: Monthly Aggregated Parent Fact Table (fact_orders)
# Full Composite Key: product_code + customer_code + date (Month Start)
# ------------------------------------------------------------------------------
parent_full_composite_keys = ["product_code", "customer_code", "date"]
parent_total = gold_parent_fact.count()

# Check for Null values in the aggregated composite key
parent_nulls = gold_parent_fact.filter(
    F.col("product_code").isNull() | 
    F.col("customer_code").isNull() | 
    F.col("date").isNull()
).count()

# Check for duplicate records across the aggregated composite key
parent_duplicates_df = (
    gold_parent_fact
    .groupBy(parent_full_composite_keys)
    .count()
    .filter(F.col("count") > 1)
)
parent_duplicates = parent_duplicates_df.count()

# Check for non-positive or null aggregated sales quantities
invalid_sales = gold_parent_fact.filter(
    F.col("sold_quantity").isNull() | (F.col("sold_quantity") <= 0)
).count()

print(f"\n[AUDIT 2] Monthly Parent Fact Table (gold.fact_{data_source}):")
print(f"  |- Total Aggregated Records: {parent_total}")
print(f"  |- Nulls in Composite Key: {parent_nulls}")
print(f"  |- Full Composite Key Duplicates ({' + '.join(parent_full_composite_keys)}): {parent_duplicates}")
print(f"  |- Invalid / Non-Positive Sales Quantities (<= 0): {invalid_sales}")

if parent_duplicates > 0:
    print("  WARNING: Duplicates detected in Monthly Parent Aggregated Table!")
    parent_duplicates_df.orderBy(F.col("count").desc()).show(5)
else:
    print("  PASSED: Monthly Parent Table is unique across all aggregated business keys.")

# ==============================================================================
# SUMMARY & FINAL VERDICT
# ==============================================================================
print("\n" + "=" * 75)
total_issues = sb_nulls + sb_duplicates + parent_nulls + parent_duplicates + invalid_sales

if total_issues == 0:
    print("FINAL RESULT: ALL GOLD LAYER TABLES PASSED FULL COMPOSITE KEY AUDIT!")
else:
    print(f"FINAL RESULT: DETECTED {total_issues} ISSUES IN GOLD TABLES. REVIEW LOGS ABOVE.")
print("=" * 75)

 🚀 STARTING FULL COMPOSITE KEY QUALITY AUDIT FOR GOLD LAYER TABLES

[AUDIT 1] Detailed Gold Fact Table (gold.sb_fact_orders):
  ├─ Total Gold Records: 40939
  ├─ Nulls in Composite Key: 0
  └─ Full Composite Key Duplicates (order_id + product_code + customer_code + date): 0
  ✅ PASSED: No true duplicates found across full composite transaction keys.

[AUDIT 2] Monthly Parent Fact Table (gold.fact_orders):
  ├─ Total Aggregated Records: 96722
  ├─ Nulls in Composite Key: 0
  ├─ Full Composite Key Duplicates (product_code + customer_code + date): 0
  └─ Invalid / Non-Positive Sales Quantities (<= 0): 0
  ✅ PASSED: Monthly Parent Table is unique across all aggregated business keys.

🎉 FINAL RESULT: ALL GOLD LAYER TABLES PASSED FULL COMPOSITE KEY AUDIT!
